# Agent 第 20 课：Cost & 性能调优 + 生产上线 checklist（Phase 2 收官）

前 9 课 Phase 2 把 Bedrock agent 的零件全讲完了。这一课不讲新概念，**把之前所有东西串成一份"能不能上线"的表**。

学完这节你能回答：
1. 一个 Bedrock agent 的钱都花在哪里、从哪里省
2. 延迟瓶颈通常在哪几个点
3. 上线前必须过的 30 条 checklist（按优先级）
4. 第 1 个月线上最先要盯的几件事

## 1. 成本结构：先搞清楚钱去哪了

一次 agent 调用，成本来自这几块：

| 成本项 | 单价（量级） | 主要影响因素 |
|---|---|---|
| **模型 input token** | 贵模型 $3-15 / M tok | instruction 长度 + tool schema + 历史 + 每次 retrieval 拼进 prompt |
| **模型 output token** | 贵模型 $15-75 / M tok | 回答长度 + rationale（trace 里 agent 自己的思考） |
| **KB retrieve** | 按 query 计费 | 每次 orchestration 决定查一次就是一笔 |
| **KB 存储 / 向量库** | OpenSearch Serverless 小时费 | OCU 最低用量 |
| **Lambda 执行** | 一般可忽略 | 高并发时要看 |
| **Guardrail** | 按 text unit | 每次调用 +一笔 |

**agent 烧钱三大漏斗**，从大到小：

1. **循环步数多**：trace 里 orchestration 每一步都是完整 re-prompt（历史全带）。5 步就是 5 次 full prompt。
2. **Tool schema / KB context 过长**：每步都重新传。挂 30 个工具的 agent 光 schema 就吃几 k token/步。
3. **模型过大**：Opus 当 router 是奢侈浪费。很多子任务 Haiku 就够。

## 2. 七个省钱动作（按 ROI 排序）

| # | 动作 | 潜在节省 | 副作用 |
|---|---|---|---|
| 1 | **选小模型做 collaborator / worker**，只 supervisor 用大模型 | 30-60% | 需 eval 对比质量 |
| 2 | **收紧 max_steps**，配合 observation 里"快速失败" | 20-40% | 少数复杂 case 不够用 → 提示用户重问 |
| 3 | **精简 tool schema description**（准而不冗） | 10-25% | 写 description 要更仔细 |
| 4 | **Prompt caching**（Anthropic 支持 — 把 instruction + tool schema 做为 cached prefix） | 40-90% 的 input 成本 | 有最少 token 门槛，小 prompt 不触发 |
| 5 | **收缩 KB chunk size / topK**：默认常常过大 | 5-15% | 召回要重新 eval |
| 6 | **用 Provisioned Throughput**：大批量稳定流量时更便宜 | 变数大 | 用量估不准反而贵 |
| 7 | **SUPERVISOR_ROUTER 而不是 SUPERVISOR**（第 19 课） | 30-50%（多 agent 场景） | 无法跨域综合 |

**一条最反直觉的**：省钱最大的往往不是换模型，而是**砍步数**。一次 5 步变 3 步，直接 40% 成本下来。

## 3. 延迟分解：瓶颈在哪

一次 invoke_agent 的 wall-clock time ≈

```
(guardrail input) 
  + Σ 每一步 orchestration:
      ├── model invoke             ← 1-5s / 步，贵模型更慢
      └── tool exec (Lambda / KB) ← 从 50ms 到几秒
  + (guardrail output)
```

**最常见的瓶颈顺序**：

1. 步数失控 —— 其他都是小头
2. Lambda 冷启动或 Lambda 内的外部 API 慢
3. 大模型 TTFT（time to first token）
4. KB retrieve 里 re-rank 慢（如果开了）

### 对应对策

| 瓶颈 | 诊断 | 对策 |
|---|---|---|
| 步数失控 | trace 里 orchestration step 数 > 5 | 改 instruction 更明确、删冗余工具、给工具更好 description |
| Lambda 冷启动 | X-Ray 看到首次调用慢、后续快 | Provisioned Concurrency，或让 Lambda 更小（无大包） |
| Lambda 内 API 慢 | X-Ray 看 subsegment | 加缓存、并发调用、超时设短 |
| 模型 TTFT | 大模型才有这问题 | 小模型 / cross-region inference profile |
| KB retrieve | trace 里 KB step 耗时 | topK ↓、chunk ↓、考虑 hybrid search 需不需要 |

## 4. 一段最小的成本统计代码

从 invoke_agent 的 trace 里提取每步 token usage：

In [ ]:
import boto3, os, uuid, json
os.environ.setdefault("AWS_REGION", "us-west-2")
runtime = boto3.client("bedrock-agent-runtime")

AGENT_ID       = "XXXXXXXXXX"
AGENT_ALIAS_ID = "YYYYYYYYYY"

# 按 2026 年粗略价（$/M token）—— 只做估算，生产要用当期价
PRICES = {
    "anthropic.claude-sonnet-4-6-20260101-v1:0": {"in": 3.0,  "out": 15.0},
    "anthropic.claude-opus-4-7-20260101-v1:0":   {"in": 15.0, "out": 75.0},
    "anthropic.claude-haiku-4-5-20251001-v1:0":  {"in": 0.25, "out": 1.25},
}

def run_and_cost(text):
    sid = str(uuid.uuid4())
    stream = runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
        sessionId=sid, inputText=text, enableTrace=True,
    )
    total_in = total_out = 0
    steps = 0
    for ev in stream["completion"]:
        if "trace" in ev:
            t = ev["trace"]["trace"]
            orch = t.get("orchestration", {})
            mio = orch.get("modelInvocationOutput", {})
            meta = mio.get("metadata", {})
            usage = meta.get("usage", {})
            if usage:
                total_in  += usage.get("inputTokens",  0)
                total_out += usage.get("outputTokens", 0)
                steps += 1
    print(f"steps={steps}, in={total_in}, out={total_out}")
    # 估算（用 supervisor 模型价；多 agent 场景要按 collaborator 模型分别算）
    p = PRICES["anthropic.claude-sonnet-4-6-20260101-v1:0"]
    usd = total_in/1_000_000*p["in"] + total_out/1_000_000*p["out"]
    print(f"est cost: ${usd:.6f}")

# run_and_cost("给我用一段话介绍向量数据库。")

## 5. 生产上线 checklist（30 条，按域）

按域归类，优先级从上到下。**少一条都算欠债。**

### A. 发布 / 运维
- [ ] 生产流量**只打 alias**，alias 指向 Version N（不是 DRAFT）
- [ ] 有回滚脚本：一行命令把 alias 指回 Version N-1
- [ ] CI/CD：PR 合并 → 自动 prepare_agent + create_version（不自动切 alias）
- [ ] 基础设施 as code（CloudFormation / CDK / Terraform），别点 console
- [ ] 有"杀死开关"：一个配置能瞬间禁用整个 agent 或某个 tool

### B. 安全
- [ ] 挂了 Bedrock Guardrails（content / denied topic / PII）—— 第 16 课
- [ ] 每个 tool 打了 danger 等级；irreversible 工具有二次确认（Return Control 或业务层）
- [ ] Action Group Lambda 的 IAM role **最小权限**（不是 *）
- [ ] 敏感工具的目标有白名单（"只能给这些 email 发"）
- [ ] 外部数据（web / KB / tool result）都视为 untrusted
- [ ] 把"红队 prompt"塞进 eval set 定期跑

### C. 可靠性
- [ ] `max_steps` 设上限（10 够多数场景）
- [ ] Lambda 超时设合理（< agent 整体 SLA）
- [ ] Lambda retry 策略明确（自动 retry + exponential backoff）
- [ ] Cross-region inference profile 开（防单 region 降级）
- [ ] 下游依赖都有超时 + 熔断（不被拖死）

### D. 观测
- [ ] invoke_agent 时 `enableTrace=True`
- [ ] Model Invocation Logging 开到 CloudWatch 或 S3
- [ ] 自定义 metrics：成功率、步数分布、工具调用分布
- [ ] 告警：成功率 < 98%、p99 > 30s、日成本 > 预算 70%
- [ ] 每次调用有 request_id 贯穿前端→agent→lambda→DB
- [ ] X-Ray 开在 Lambda

### E. 成本
- [ ] 用对模型：supervisor 用大、collaborator 用小
- [ ] Prompt caching 开（Anthropic 模型；把 instruction + schema 做前缀）
- [ ] KB topK / chunk 参数经过 eval 调过（不是默认）
- [ ] 有日成本面板，每个 agent 单独看
- [ ] 预算告警（AWS Budgets）

### F. 数据 / 合规
- [ ] KB 数据来源有权限控制（不是全公司可写）
- [ ] Model Invocation Log 权限收紧 + 保留期合理
- [ ] 用户能"请求删除" agent memory（`delete_agent_memory`）
- [ ] PII 进 agent 前过 Guardrail ANONYMIZE，或业务层脱敏
- [ ] 多租户隔离：sessionAttributes 里带 tenant_id，Lambda 强校验

### G. 质量 / eval
- [ ] 有 golden set（50-200 条），每次发布前跑
- [ ] 指标至少有：成功率、平均步数、平均 token、平均延迟
- [ ] 换模型 / 改 instruction 时要对比两组指标（A/B 或离线）
- [ ] routing eval（multi-agent 场景）单独有

## 6. 发布后第 1 个月盯什么

上线不是终点，是观察的起点。头 4 周每周至少看一次这 5 件事：

1. **成本曲线** vs 预估：超 30% 必须查原因（通常是步数失控 或 一个 tool 被反复调）
2. **成功率分布**：按 tenant / 按问题类型切。整体 99% 不代表某个关键用户群也 99%
3. **步数 p50 / p95**：p95 突增往往是新一类查询触发了死循环
4. **Guardrail 命中率**：骤升可能是被滥用或注入攻击；骤降可能是策略被绕过
5. **用户反馈 / rating**：线上最真的 eval。attach 到 sessionId，能回放 trace

### 迭代节奏建议

- **每周**：看 metrics 大盘 + 抽 20 个失败 case 人工过
- **每月**：golden set 扩一轮，淘汰老的 / 加新场景
- **每季**：模型重评 —— 有没有更便宜 / 更快 / 更好的新模型出来

## 7. 工程直觉（收官一刻钟）

1. **省钱从砍步数开始**，不是换模型。trace 里 > 5 步的请求占比是第一可观测指标。
2. **Prompt caching 是 Bedrock + Anthropic 的免费午餐**。把 instruction + tool schema 写稳，剩下的就是白捡。
3. **Alias 是生产唯一入口**、**trace 默认开**、**每次请求有 request_id** —— 这三条是上线的地基，缺一个就别上。
4. **Eval 是 agent 工程最没人爱做、但 ROI 最高的一件事**。一套能跑的 golden set 比多 10 个新功能都救命。
5. **Agent 不是越聪明越好**。能用规则写死就别让 LLM 判断；能用一个 single agent 就别拆多 agent。复杂度的成本都是你的。
6. **Phase 1 的那些手写技能从来没白学**。托管服务的每一层抽象，你都能一眼看穿它包的是什么、在 Phase 1 的哪一课讲过。出事时你知道从哪扒。

---

## Phase 2 完结 🎉

一路从"什么是 agent"到"生产 agent 上线 checklist"。Phase 1 让你懂原理，Phase 2 让你能交付。

**下一步往哪走**：
- 选一个你关心的真实业务场景，从 0 搭一个 Bedrock agent，过一遍这份 checklist
- 读 Anthropic [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) 和 [Claude SDK](https://docs.claude.com/en/docs/claude-code/sdk)——开源生态的另一条路
- 把自己踩过的坑写回这本 notebook，这才是你的
